# CEP Sim — UK
Brand recall simulator for UK beer market (Heineken study, N=1,326).

**Brands:** Heineken, Guinness, Stella Artois, Budweiser, Corona Extra, Madri, and 17 others  
**CEPs:** 21 purchase occasions (Q10–Q30) in English  
**Config:** `backend/configs/cep_sim_config_uk.toml`


In [ ]:
import sys
sys.path.insert(0, '../../../..')  # project root

import logging
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s - %(message)s')


## 1. Load config

In [ ]:
from pathlib import Path
from backend.schemas.config import load_cep_sim_config

PROJECT_ROOT = Path('../../../../').resolve()

config = load_cep_sim_config(PROJECT_ROOT / 'backend/configs/cep_sim_config_uk.toml')

# Resolve relative paths to absolute
config.survey.zip_path      = str(PROJECT_ROOT / config.survey.zip_path)
config.output.outputs_dir   = str(PROJECT_ROOT / config.output.outputs_dir)
config.output.processed_dir = str(PROJECT_ROOT / config.output.processed_dir)

print('Country   :', config.survey.country)
print('CEP blocks:', config.survey.recall.cep_blocks)
print('Outputs   :', config.output.outputs_dir)

## 2. Load raw survey

In [ ]:
from backend.service.load_data import load_survey, inspect_survey

df = load_survey(config)
info = inspect_survey(df, config)
print(f"{info['row_count']} respondents  |  {info['recall_column_count']} recall columns  |  {len(info['cep_blocks'])} CEP blocks")


## 3. Reshape wide → long

In [ ]:
from backend.service.reshape_survey import reshape_wide_to_long, save_long_survey

long_df = reshape_wide_to_long(df, config)
save_long_survey(long_df, config)

print(f"Long table : {len(long_df):,} rows")
print(f"Unique CEPs: {long_df['cep_description'].nunique()}")
print(f"Mention rate: {long_df['mentioned'].mean():.1%}")
long_df.head(3)


## 4. Build CEP ontology

In [ ]:
from backend.service.ontology_builder import build_ontology, save_ontology

cep_master_df, raw_map_df = build_ontology(long_df, config)
save_ontology(cep_master_df, raw_map_df, config)

print(f"{len(cep_master_df)} CEPs across families:")
print(cep_master_df.groupby('cep_family').size().sort_values(ascending=False).to_string())
cep_master_df[['cep_id','cep_family','cep_label']]


## 5. Build respondent memory tables

In [ ]:
from backend.service.respondent_builder import (
    build_respondents, build_respondent_brand_cep, save_respondent_tables
)

respondents_df = build_respondents(df, config)
rbc_df = build_respondent_brand_cep(long_df, raw_map_df, config)
save_respondent_tables(respondents_df, rbc_df, config)

print(f"{len(respondents_df):,} respondents  |  {len(rbc_df):,} memory edges")
print(f"Segments: {respondents_df['segment'].value_counts().head(5).to_dict()}")

brand_name_map = rbc_df[['brand_id','brand_name']].drop_duplicates().set_index('brand_id')['brand_name'].to_dict()
respondent_ids = respondents_df['respondent_id'].astype(str).tolist()

top = rbc_df.groupby('brand_name').size().sort_values(ascending=False).head(8)
print("\nTop 8 brands by memory edges:")
print(top.to_string())


## 6. Baseline recall — single respondent

In [ ]:
from backend.service.recall_engine import get_recall_probs, rank_brands, get_scenarios

SCENARIOS = get_scenarios('UK')
sample_id = str(respondents_df['respondent_id'].iloc[0])
scenario  = next(s for s in SCENARIOS if s['scenario_name'] == 'trendy_bar')

probs  = get_recall_probs(sample_id, scenario['active_ceps'], rbc_df, cep_master_df, config)
ranked = rank_brands(probs)

print(f"Respondent {sample_id} | Scenario: {scenario['scenario_name']}")
print(f"{'Rank':<5} {'Brand':<28} {'Prob':>6}")
for rank, (brand_id, prob) in enumerate(ranked[:8], 1):
    print(f"{rank:<5} {brand_name_map.get(brand_id, brand_id):<28} {prob:.4f}")


## 7. Baseline recall — all respondents × all scenarios

In [ ]:
from backend.service.validator import run_scenario_recall

scenario_recall_df = run_scenario_recall(
    respondent_ids, SCENARIOS, rbc_df, cep_master_df, brand_name_map, config
)
print(f"Scenario recall: {len(scenario_recall_df):,} rows")

# Average recall prob — trendy bar
trendy = scenario_recall_df[scenario_recall_df['scenario_name'] == 'trendy_bar']
avg    = trendy.groupby('brand_name')['recall_prob'].mean().sort_values(ascending=False).head(8)

fig, ax = plt.subplots(figsize=(9, 4))
avg.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('UK — Avg Recall Probability: Trendy Bar')
ax.set_ylabel('Avg Recall Prob')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()


## 8. Calibration check

In [ ]:
from backend.service.validator import run_calibration_check

cal_df = run_calibration_check(scenario_recall_df, long_df)
print(f"MAE: {cal_df.attrs['mae']:.4f}  (mean |predicted recall_prob − observed mention rate|)")
cal_df


## 9. Define Heineken ad and apply it

In [ ]:
from backend.service.ad_engine import Ad, apply_ad_to_population, save_episodic_events

# UK ad: target trendy bar (focal) + live music gig (secondary)
trendy_bar_cep = cep_master_df[
    cep_master_df['cep_description'].str.contains('trendy bar', case=False, na=False)
]['cep_id'].values[0]

gig_cep = cep_master_df[
    cep_master_df['cep_description'].str.contains('live music', case=False, na=False)
]['cep_id'].values[0]

heineken_ad = Ad(
    ad_id='ad_heineken_uk_trendy_bar_001',
    brand_id='brand_heineken',
    brand_name='Heineken',
    focal_ceps=[trendy_bar_cep],
    secondary_ceps=[gig_cep],
    branding_clarity=0.9,
    attention_weight=1.0,
    channel='social_media',
    emotion='confidence',
)

rbc_post, events = apply_ad_to_population(respondent_ids, heineken_ad, rbc_df, config)
save_episodic_events(events, config)

print(f"Ad: {heineken_ad.ad_id}")
print(f"Focal CEP   : {trendy_bar_cep}")
print(f"Secondary   : {gig_cep}")
print(f"Applied to  : {len(events)} respondents")
print(f"Edges pre   : {len(rbc_df):,}")
print(f"Edges post  : {len(rbc_post):,}")


## 10. Before vs after recall

In [ ]:
from backend.service.validator import run_ad_impact, build_segment_summary

ad_impact_df = run_ad_impact(
    respondent_ids, SCENARIOS, rbc_df, rbc_post, cep_master_df, brand_name_map, config
)

impact = (
    ad_impact_df[ad_impact_df['scenario_name'] == 'trendy_bar']
    .groupby('brand_name')[['recall_pre','recall_post','delta']]
    .mean().round(4)
    .sort_values('delta', ascending=False)
    .head(8)
)
print("UK — Avg recall shift: Trendy Bar (Heineken ad)")
print(impact.to_string())


In [ ]:
top8 = impact.head(8).index
plot_df = impact.loc[top8][['recall_pre','recall_post']]

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(plot_df))
ax.bar([i - 0.2 for i in x], plot_df['recall_pre'],  width=0.4, label='Pre',  color='steelblue')
ax.bar([i + 0.2 for i in x], plot_df['recall_post'], width=0.4, label='Post', color='coral')
ax.set_xticks(list(x)); ax.set_xticklabels(plot_df.index, rotation=35, ha='right')
ax.set_title('UK — Recall Pre vs Post: Trendy Bar (Heineken ad)')
ax.set_ylabel('Avg Recall Score'); ax.legend()
plt.tight_layout(); plt.show()


## 11. Segment summary

In [ ]:
segment_summary_df = build_segment_summary(ad_impact_df, respondents_df)
print(f"Segment summary: {len(segment_summary_df):,} rows")
segment_summary_df[segment_summary_df['scenario_name'] == 'trendy_bar'].head(12)


## 12. Export + sanity checks

In [ ]:
from backend.service.validator import save_outputs, run_sanity_checks

paths = save_outputs(scenario_recall_df, ad_impact_df, segment_summary_df, config)
for name, path in paths.items():
    print(f"  {name}: {path}")

print("\n=== Sanity Checks ===")
for k, v in run_sanity_checks(scenario_recall_df, ad_impact_df).items():
    print(f"  {k}: {v}")


## 13. Generate standard outputs (artifact contract)

In [ ]:
from backend.service.output_builder import generate_standard_outputs

saved = generate_standard_outputs(
    rbc_pre=rbc_df,
    rbc_post=rbc_post,
    impact_df=ad_impact_df,
    cep_master_df=cep_master_df,
    long_df=long_df,
    scenario_recall_df=scenario_recall_df,
    focal_brand_id=heineken_ad.brand_id,
    focal_brand_name=heineken_ad.brand_name,
    focal_scenario='trendy_bar',
    config=config,
    segment_summary_df=segment_summary_df,
)

print(f"{len(saved)} files written:")
for name, path in saved.items():
    print(f"  {name}: {path}")
